# **Sesión 2:** Calibración de Cámara 📷⚙️

## 0. Preparación

In [1]:
import cv2
print("OpenCV should be 4.8.0.76 Current version:", cv2.__version__)
from typing import List
import numpy as np
import imageio
import cv2
import copy
import glob

OpenCV should be 4.8.0.76 Current version: 4.8.0


## **Apartado A: Calibración de cámara** (derecha e izquierda)

En este apartado se realiza la calibración de dos cámaras de un sistema estereoscópico. Para ello se trabajará con las imágenes de las carpetas ``left`` y ``right``. En primer lugar se trabajará con la carpeta ``left``. Posteriormente, deberá repetir el proceso con las imágenes en la carpeta ``right``. Ambas carpetas contienen imágenes con las que se calibrarán las cámaras. En ellas aparece el patrón de calibración en diferentes posiciones y orientaciones. Estas imágenes serán los datos de entrada.

Los pasos que deberá seguir para calibrar una cámara son:

1. Defina y ejecute el método para cargar imágenes ``load_images()``.
2. Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.
3. Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``.
4. Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)`` que proporcione las coordenadas 3D de las esquinas del patrón. El sistema de referencia utilizado deberá estar anclado al propio patrón.
5. Utilice ``cv2.calibrateCamera`` para obtener los parámetros de calibración para la cámara izquierda.

### **Tarea A.1:** Defina y ejecute el método para cargar imágenes ``load_images()``.

In [2]:
def load_images(filenames: List) -> List:
    return [imageio.imread(filename) for filename in filenames]

In [3]:
# TODO Build a list containing the paths of all images from the left camera

import glob  # librería necesaria para buscar nombres de paths con patrones concretos, usando *, etc.

imgs_gen_path = '../data/left/*.jpg' # patrón que cumplen todos los paths de los archivos de imágenes
imgs_path = sorted(glob.glob(imgs_gen_path))  # cogemos todos los paths que cumplen ese patrón genérico (usando glob) y los ordenamos (usando sorted)
imgs = load_images(imgs_path)  # le pasamos a load_images la lista de paths o filenames de las imágenes que queremos cargar

imgs_path

C:\Users\Estudios\AppData\Local\Temp\ipykernel_13304\283608422.py:2: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  return [imageio.imread(filename) for filename in filenames]


['../data/left\\left_000.jpg',
 '../data/left\\left_001.jpg',
 '../data/left\\left_002.jpg',
 '../data/left\\left_003.jpg',
 '../data/left\\left_004.jpg',
 '../data/left\\left_005.jpg',
 '../data/left\\left_006.jpg',
 '../data/left\\left_007.jpg',
 '../data/left\\left_008.jpg',
 '../data/left\\left_009.jpg',
 '../data/left\\left_010.jpg',
 '../data/left\\left_011.jpg',
 '../data/left\\left_012.jpg',
 '../data/left\\left_013.jpg',
 '../data/left\\left_014.jpg',
 '../data/left\\left_015.jpg',
 '../data/left\\left_016.jpg',
 '../data/left\\left_017.jpg',
 '../data/left\\left_018.jpg']

### **Tarea A.2:** Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.

In [4]:
# contamos a mano cuantas esquinas de filas y columnas hay en el tablero de chess
pattern_size = (8,6)

In [5]:
# TODO Find corners with cv2.findChessboardCorners()
corners = [cv2.findChessboardCorners(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), pattern_size) for img in imgs]

In [6]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

# TODO To refine corner detections with cv2.cornerSubPix() you need to input grayscale images. Build a list containing grayscale images.
imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs] # pasamos a grises las imágenes

# esto quita los corners de las imagenes que no hemos podido calibrar bien
corners_refined = [cv2.cornerSubPix(i, cor[1], (8, 6), (-1, -1), criteria) if cor[0] else [] for i, cor in zip(imgs_gray, corners_copy)]

### **Tarea A.3:** Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``

In [7]:
# quitamos la primera imagen ya que esa no la hemos podido calibrar
imgs_copy = copy.deepcopy(imgs) 


In [8]:
# TODO Use cv2.drawChessboardCorners() to draw the cornes

for i in range(len(corners_copy)):
    # solo dibujamos las que tienen True en ret
    ret = corners_copy[i][0]
    if ret:
        cv2.drawChessboardCorners(imgs_copy[i], pattern_size, corners_refined[i], corners_copy[i][0])
        cv2.imshow("Img",imgs_copy[i])
        cv2.waitKey(0)

cv2.destroyAllWindows()

In [9]:
cv2.imshow("Img",imgs_copy[0])
cv2.waitKey(0)
cv2.destroyAllWindows()

In [10]:
import os

# TODO Complete the method, use every argument
def show_image(img: np.array, img_name: str = "Image"):
    cv2.imshow(img_name, img)  # mostramos la imagen, poniendole el nombre correspondiente a la ventana desplegada
    cv2.waitKey(0)  # ponemos una tecla de introducción
    cv2.destroyAllWindows()

# TODO Complete the method, use every argument
def write_image(output_folder: str, img_name: str, img: np.array):
    img_path = os.path.join(output_folder, img_name)  # creamos el path donde vamos a guardar la imagen
    os.makedirs(output_folder, exist_ok=True)
    cv2.imwrite(img_path, img)  # guardamos la imagen en su correspondiente ubicación

# we save all the images of ther corners
for i in range(len(imgs_copy)):
    write_image("imagenes_creadas", f"patron_{i}.jpg", imgs_copy[i])

### **Tarea A.4:** Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)``

In [11]:
import numpy as np

# TODO Design the method. It should return a np.array with np.float32 elements
def get_chessboard_points(chessboard_shape, dx, dy):

    # to do comments
    cols, rows = chessboard_shape

    grid_xy = np.mgrid[0:cols, 0:rows].T.reshape(-1,2)

    objp = np.zeros([grid_xy.shape[0], 3], dtype=np.float32)
    objp[:,0] = grid_xy[:,0]*dx
    objp[:,1] = grid_xy[:,1]*dy

    return objp

In [12]:
# TODO You need the points for every image, not just one (consider a list comprehension)
chessboard_points = [get_chessboard_points((8, 6), 30, 30) for _ in corners_refined]

### **Tarea A.5:** Utilice ``cv2.calibrateCamera()`` para obtener los parámetros de calibración para la cámara izquierda

In [13]:
# Filter data and get only those with adequate detections
valid_corners = [cor[1] for cor in corners if cor[0]]
# Convert list to numpy array
valid_corners = np.asarray(valid_corners, dtype=np.float32)

In [14]:
# TODO
# valid son los reales y chessboard_points son los estimados
img_size = (imgs_gray[0].shape[1], imgs_gray[0].shape[0])
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(chessboard_points, valid_corners, img_size, None, None)

# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))

In [15]:
# Print outputs
print("Intrinsics:\n", intrinsics)
print("Distortion coefficients:\n", dist_coeffs)
print("Root mean squared reprojection error:\n", rms)

Intrinsics:
 [[417.77315866   0.         156.33829983]
 [  0.         421.35480199 134.32880063]
 [  0.           0.           1.        ]]
Distortion coefficients:
 [[-2.13049815e-01  2.86830664e+00  5.90990527e-03 -3.86453734e-03
  -2.23696363e+01]]
Root mean squared reprojection error:
 0.1846287136881559


### **Pregunta A.1:** Repita el proceso (carga de imágenes, detección y comprobación de esquinas, etc.) para la cámara derecha.

In [16]:
# TODO Homework
# TODO Build a list containing the paths of all images from the left camera

import glob  # librería necesaria para buscar nombres de paths con patrones concretos, usando *, etc.

imgs_gen_path = '../data/right/*.jpg' # patrón que cumplen todos los paths de los archivos de imágenes
imgs_path = sorted(glob.glob(imgs_gen_path))  # cogemos todos los paths que cumplen ese patrón genérico (usando glob) y los ordenamos (usando sorted)
imgs = load_images(imgs_path)  # le pasamos a load_images la lista de paths o filenames de las imágenes que queremos cargar

imgs_path

C:\Users\Estudios\AppData\Local\Temp\ipykernel_13304\283608422.py:2: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  return [imageio.imread(filename) for filename in filenames]


['../data/right\\right_000.jpg',
 '../data/right\\right_001.jpg',
 '../data/right\\right_002.jpg',
 '../data/right\\right_003.jpg',
 '../data/right\\right_004.jpg',
 '../data/right\\right_005.jpg',
 '../data/right\\right_006.jpg',
 '../data/right\\right_007.jpg',
 '../data/right\\right_008.jpg',
 '../data/right\\right_009.jpg',
 '../data/right\\right_010.jpg',
 '../data/right\\right_011.jpg',
 '../data/right\\right_012.jpg',
 '../data/right\\right_013.jpg',
 '../data/right\\right_014.jpg',
 '../data/right\\right_015.jpg',
 '../data/right\\right_016.jpg',
 '../data/right\\right_017.jpg',
 '../data/right\\right_018.jpg']

### **Pregunta A.2:** Diferencias entre cv2.findChessboardCorners() y cv2.cornerSubPix()

In [17]:
# TODO Homework
# Te da el pixel exacto el que mayor gradiente tiene, el subpix nos lo da con decimales, ponderando te dice cual es más preciso
# el subpix sirve para perfeccionar porque mira los gradientes y es más preciso a la hora de buscar el punto corner, nos da coordenadas de un "subpixel"
# mejora el error de proyección
"""

idea para nota:

explicar diferencias y poner una imagen, con los corners buscados solo con findChessboardCorners y otra con cornerSubPix
y luego tmbn los RMSE



"""

'\n\nidea para nota:\n\nexplicar diferencias y poner una imagen, con los corners buscados solo con findChessboardCorners y otra con cornerSubPix\ny luego tmbn los RMSE\n\n\n\n'

### **Pregunta A.3:** Número mínimo de imágenes necesarias para calibrar.

In [18]:
# TODO Homework

## **Apartado B: Corrección de distorsión** (ojo de pez)

En este apartado se trabajará en la corrección de la distorsión debido a lentes de ojo de pez. Primero se calibrará una cámara con este tipo de lente, utilizando las imágenes de la carpeta ``fisheye``. Posteriormente se utilizarán estos parámetros de calibración para corregir la distorsión de una de las imágenes de calibración.

Los pasos que deberá seguir para calibrar una cámara con distorsión de ojo de pez son:

1. Reutilice el método ``load_images()`` para cargar las imágenes de la carpeta ``fisheye``.
2. Detecte las equinas procesando las imágenes con los métodos ``cv2.findChessboardCorners()`` y ``cv2.cornerSubPix()``.
3. Reutilice la función ``get_chessboard_points()`` para obtener las coordenadas del tablero.
4. Defina los argumentos para la función de calibración.
5. Calibre con ``cv2.fisheye.calibrate()``

### **Tarea B.1:** Reutilice el método ``load_images()`` para cargar las imágenes de la carpeta ``fisheye``

In [41]:
# TODO Build a list containing the paths of all images from the fisheye camera and load images

fisheye_imgs_path = '../data/fisheye/*.jpg' # patrón que cumplen todos los paths de los archivos de imágenes
imgs_fisheye_path = sorted(glob.glob(fisheye_imgs_path))  # cogemos todos los paths que cumplen ese patrón genérico (usando glob) y los ordenamos (usando sorted)
fisheye_imgs = load_images(imgs_fisheye_path)  # le pasamos a load_images la lista de paths o filenames de las imágenes que queremos cargar

imgs_fisheye_path

C:\Users\Estudios\AppData\Local\Temp\ipykernel_13304\283608422.py:2: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  return [imageio.imread(filename) for filename in filenames]


['../data/fisheye\\VMRImage0.jpg',
 '../data/fisheye\\VMRImage1.jpg',
 '../data/fisheye\\VMRImage2.jpg',
 '../data/fisheye\\VMRImage3.jpg',
 '../data/fisheye\\VMRImage4.jpg',
 '../data/fisheye\\VMRImage5.jpg',
 '../data/fisheye\\VMRImage6.jpg',
 '../data/fisheye\\VMRImage7.jpg',
 '../data/fisheye\\VMRImage8.jpg',
 '../data/fisheye\\VMRImage9.jpg']

### **Tarea B.2:** Detecte las equinas procesando las imágenes con los métodos ``cv2.findChessboardCorners()`` y ``cv2.cornerSubPix()``.

In [42]:
fisheye_imgs_copy = copy.deepcopy(fisheye_imgs)
fisheye_size_pattern = (7,6)

imgs_corners = []
# Parameters for cv2.cornerSubPix()
subpix_criteria = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1)

#TODO Complete the required parts of the loop
for img in fisheye_imgs_copy:
    
    # TODO parse arguments to cv2.findChessboardCorners()
    corners = cv2.findChessboardCorners(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), fisheye_size_pattern)
    
    # TODO convert image to grayscale to use cv2.cornerSubPix()
    gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    if corners[0]:  # only refine if corners have been found
        refined_corners = cv2.cornerSubPix(gray_img, corners[1], (3,3), (-1,-1), subpix_criteria)

        # TODO append only those refined_corners with proper detections
        imgs_corners.append(refined_corners)

In [48]:
for i in range(len(imgs_corners)):
    cv2.drawChessboardCorners(fisheye_imgs_copy[i], fisheye_size_pattern, imgs_corners[i], True)
    cv2.imshow("Img",fisheye_imgs_copy[i])
    cv2.waitKey(0)
    cv2.destroyAllWindows()

### **Tarea B.3:** Reutilice la función ``get_chessboard_points()`` para obtener las coordenadas del tablero

In [51]:
# TODO Define the chessboard dimensions and the lenght of the squares (in [mm])
chessboard_dims = (7, 6)
length = 30
# TODO You need the points for every image, not just one (consider a list comprehension)
fisheye_chessboard_points = [get_chessboard_points(chessboard_dims, length, length).reshape(-1,1,3) for _ in imgs_corners]
print(fisheye_chessboard_points)

[array([[[  0.,   0.,   0.]],

       [[ 30.,   0.,   0.]],

       [[ 60.,   0.,   0.]],

       [[ 90.,   0.,   0.]],

       [[120.,   0.,   0.]],

       [[150.,   0.,   0.]],

       [[180.,   0.,   0.]],

       [[  0.,  30.,   0.]],

       [[ 30.,  30.,   0.]],

       [[ 60.,  30.,   0.]],

       [[ 90.,  30.,   0.]],

       [[120.,  30.,   0.]],

       [[150.,  30.,   0.]],

       [[180.,  30.,   0.]],

       [[  0.,  60.,   0.]],

       [[ 30.,  60.,   0.]],

       [[ 60.,  60.,   0.]],

       [[ 90.,  60.,   0.]],

       [[120.,  60.,   0.]],

       [[150.,  60.,   0.]],

       [[180.,  60.,   0.]],

       [[  0.,  90.,   0.]],

       [[ 30.,  90.,   0.]],

       [[ 60.,  90.,   0.]],

       [[ 90.,  90.,   0.]],

       [[120.,  90.,   0.]],

       [[150.,  90.,   0.]],

       [[180.,  90.,   0.]],

       [[  0., 120.,   0.]],

       [[ 30., 120.,   0.]],

       [[ 60., 120.,   0.]],

       [[ 90., 120.,   0.]],

       [[120., 120.,   0.]],

       [[

### **Tarea B.4:** Defina los argumentos para la calibración

In [52]:
# Parameters for cv2.fisheye.calibrate()
calibration_flags = cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC+cv2.fisheye.CALIB_FIX_SKEW
intrinsics = np.zeros((3, 3))
distortion = np.zeros((4, 1))
rotations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]
traslations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]


### **Tarea B.5:** Calibración

In [53]:
rms, _, _, _, _ = \
cv2.fisheye.calibrate(fisheye_chessboard_points, imgs_corners, gray_img.shape[::-1], intrinsics, distortion, rotations, traslations, calibration_flags, subpix_criteria)

In [54]:
# Show intrinsic matrix and distortion coefficients values
print(intrinsics)
print(distortion)

[[188.86914026   0.         503.92900347]
 [  0.         185.32638678 373.58044468]
 [  0.           0.           1.        ]]
[[ 0.07734388]
 [ 0.01482361]
 [ 0.02351869]
 [-0.02757565]]


### **Pregunta B.1:** Corrija la distorsión de las 2 primeras imágenes de la carpeta ``fisheye``

In [ ]:
# TODO Search in the documentation to define 'dim'
dim = None
map1, map2 = cv2.fisheye.initUndistortRectifyMap(intrinsics, distortion, np.eye(3), intrinsics, dim, cv2.CV_16SC2)

In [ ]:
# TODO Homework: correct distortion using cv2.remap()